# Notebook 30 — IMERG catalog plan and Earthdata login

This is Phase 1 of the precipitation analysis. It does not download ERA5 or IMERG data. It creates a permanent Drive inventory of every precipitation window requested from the expanded merged catalog, and provides one explicit NASA Earthdata login cell.

The predictor is already available: the saved 12-hour divergence at each peak of the digitized Shinoda JPCZ polygon. Its negative is used as convergence strength, so larger positive values mean stronger convergence.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git'
BRANCH = 'codex/notebook16-pcolormesh'
REPO_DIR = '/content/JPCZcatalog'
FORCE_REFRESH_REPO = True
DRIVE_ROOT = Path('/content/drive/MyDrive/JPCZcatalog_outputs')
# Accept a raw Git URL even if it was accidentally pasted as a Markdown link.
if REPO_URL.startswith('[') and '](' in REPO_URL and REPO_URL.endswith(')'):
    REPO_URL = REPO_URL.rsplit('](', 1)[1][:-1]
if not REPO_URL.startswith('https://'):
    raise ValueError(f'REPO_URL must be a raw https Git URL, not {REPO_URL!r}')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
if FORCE_REFRESH_REPO and Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
if not Path(REPO_DIR).exists():
    clone = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], text=True, capture_output=True)
    if clone.returncode:
        raise RuntimeError(f'Git clone failed for {REPO_URL}:\n{clone.stderr}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR], check=True)
os.chdir(REPO_DIR)
if f'{REPO_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/src')
print('Repository branch:', BRANCH)
print('Drive root:', DRIVE_ROOT)

In [ ]:
import pandas as pd

from jpcz_catalog.imerg_workflow import IMERG_FIRST_VALID_TIME, IMERG_FINAL_V07_END_EXCLUSIVE, atomic_csv, prepare_imerg_event_catalog, read_checkpoint, write_event_plan

CATALOG_OVERRIDE_PATH = None
DRIVE_ANALYSIS_DIR = DRIVE_ROOT / 'imerg_precipitation_convergence'
DRIVE_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
IMERG_EVENT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_regional_precipitation.csv'
PLAN_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_collection_plan.csv'
MANIFEST_PATH = DRIVE_ANALYSIS_DIR / 'imerg_catalog_run_manifest.csv'
EXCLUSIONS_PATH = DRIVE_ANALYSIS_DIR / 'imerg_final_v07_unavailable_events.csv'

candidates = [
    Path(CATALOG_OVERRIDE_PATH) if CATALOG_OVERRIDE_PATH else None,
    DRIVE_ROOT / 'jpcz_catalog_ndjf_merged_12h.csv',
    Path('outputs/verification/jpcz_catalog_ndjf_merged_12h.csv'),
]
catalog_path = next((path for path in candidates if path is not None and path.exists()), None)
if catalog_path is None:
    raise FileNotFoundError('No merged catalog found. Run Notebook 06 first.')

catalog = pd.read_csv(catalog_path)
events = prepare_imerg_event_catalog(catalog)
existing_metrics = read_checkpoint(IMERG_EVENT_PATH, parse_dates=('event_peak',))
plan = write_event_plan(events, existing_metrics, path=PLAN_PATH)
final_v07_exclusions = plan.loc[plan['analysis_inclusion'].eq('exclude')].copy()
atomic_csv(final_v07_exclusions, EXCLUSIONS_PATH)
manifest = pd.DataFrame([{
    'catalog_source': str(catalog_path), 'merged_catalog_rows': len(catalog),
    'imerg_eligible_events': len(events), 'excluded_before_imerg': len(catalog) - len(events),
    'final_v07_end_exclusive_utc': IMERG_FINAL_V07_END_EXCLUSIVE,
    'excluded_after_final_v07_coverage': len(final_v07_exclusions),
    'first_event_peak_utc': events['event_peak'].min(), 'last_event_peak_utc': events['event_peak'].max(),
    'predictor': 'saved catalogued Shinoda-polygon -D12 convergence',
}])
atomic_csv(manifest, MANIFEST_PATH)

print('Catalog selected:', catalog_path)
print(f'IMERG event plan: {len(events)} eligible events; {len(catalog) - len(events)} excluded before IMERG coverage.')
print(f"Final V07 availability: {(plan.analysis_inclusion == 'include').sum()} included; {len(final_v07_exclusions)} excluded after the V07 Final archive cutoff ({IMERG_FINAL_V07_END_EXCLUSIVE:%Y-%m-%d}).")
print('Plan saved:', PLAN_PATH)
display(plan['collection_status'].value_counts().rename_axis('status').reset_index(name='event_count'))
display(plan.head())
if not final_v07_exclusions.empty:
    print('Documented Final-V07 exclusions saved:', EXCLUSIONS_PATH)
    display(final_v07_exclusions[['event_id', 'event_start', 'event_end', 'event_peak', 'collection_status']])

## Authenticate with NASA Earthdata

Run this one cell once per Colab session. Earthdata manages the credential flow; the notebook never stores a username or password. If authentication succeeds, continue in Notebook 31.

In [ ]:
import earthaccess
earthaccess.login()
print('Earthdata authentication completed for this Colab session. Next: Notebook 31 data collection.')